MicroScan measurement values

In [ ]:
import pandas as pd

raw_microScan = pd.read_csv(
    "/groups/ds/Win-KID/BVBRC/BVBRC_devices/input/MicroScan_combined.csv", sep=";"
)
samples_microScan = pd.read_csv(
    "/groups/ds/Win-KID/BVBRC/BVBRC_devices/git/stackPredAMR_Training_Data_Composition/Experiment_1/input_data/microScan_filtered.csv"
)

raw_microScan_filtered = raw_microScan[
    raw_microScan["Genome ID"].isin(samples_microScan["Sample_ID_IfH"])
]

raw_microScan_filtered = raw_microScan_filtered[
    raw_microScan_filtered["Antibiotic"].str.lower() == "tobramycin"
].reset_index(drop=True)

print(raw_microScan_filtered["Measurement Value"].value_counts().sort_index())

Phoenix measurement values

In [ ]:
import pandas as pd

raw_phoenix = pd.read_csv(
    "/groups/ds/Win-KID/BVBRC/BVBRC_devices/input/phoenix_combined.csv", sep=";"
)
samples_microScan = pd.read_csv(
    "/groups/ds/Win-KID/BVBRC/BVBRC_devices/git/stackPredAMR_Training_Data_Composition/Experiment_1/input_data/microScan_filtered.csv"
)

raw_phoenix_filtered = raw_phoenix[
    raw_phoenix["Genome ID"].isin(samples_microScan["Sample_ID_IfH"])
]

raw_phoenix_filtered = raw_phoenix_filtered[
    raw_phoenix_filtered["Antibiotic"].str.lower() == "tobramycin"
].reset_index(drop=True)

print(raw_phoenix_filtered["Measurement Value"].value_counts().sort_index())

VITEK measurement values

In [ ]:
import pandas as pd

raw_VITEK = pd.read_csv(
    "/groups/ds/Win-KID/BVBRC/BVBRC_devices/input/VITEK_combined.csv", sep=";"
)
samples_microScan = pd.read_csv(
    "/groups/ds/Win-KID/BVBRC/BVBRC_devices/git/stackPredAMR_Training_Data_Composition/Experiment_1/input_data/microScan_filtered.csv"
)

raw_VITEK_filtered = raw_VITEK[
    raw_VITEK["Genome ID"].isin(samples_microScan["Sample_ID_IfH"])
]

raw_VITEK_filtered = raw_VITEK_filtered[
    raw_VITEK_filtered["Antibiotic"].str.lower() == "tobramycin"
].reset_index(drop=True)

print(raw_VITEK_filtered["Measurement Value"].value_counts().sort_index())

Phoenix test data measurement values

In [ ]:
raw_phoenix = pd.read_csv(
    "/groups/ds/Win-KID/BVBRC/BVBRC_devices/input/phoenix_combined.csv", sep=";"
)
samples_phoenix_extra = pd.read_csv(
    "/groups/ds/Win-KID/BVBRC/BVBRC_devices/git/stackPredAMR_Training_Data_Composition/Experiment_1/input_data/phoenix_extra.csv"
)

raw_phoenix_filtered = raw_phoenix[
    raw_phoenix["Genome ID"].isin(samples_phoenix_extra["Sample_ID_IfH"])
]

raw_phoenix_filtered = raw_phoenix_filtered[
    raw_phoenix_filtered["Antibiotic"].str.lower() == "tobramycin"
].reset_index(drop=True)

print(raw_phoenix_filtered["Measurement Value"].value_counts().sort_index())

device-pair susceptibility confusion matrices

In [3]:
import pandas as pd

samples_phoenix = pd.read_csv("phoenix_filtered.csv")
samples_vitek = pd.read_csv("vitek_filtered.csv")
samples_microScan = pd.read_csv("microScan_filtered.csv")

devices = {
    "Phoenix": samples_phoenix,
    "VITEK": samples_vitek,
    "MicroScan": samples_microScan,
}

labels = {}
for device, df in devices.items():
    data = df[["Sample_ID_IfH", "tobramycin"]].copy()
    data["Sample_ID_IfH"] = data["Sample_ID_IfH"].astype("string").str.strip()
    data["tobramycin"] = data["tobramycin"].astype("string").str.strip().str.upper()

    valid = (
        data["Sample_ID_IfH"].notna()
        & data["Sample_ID_IfH"].ne("")
        & data["tobramycin"].isin(["S", "R"])
    )
    # Identische Wiederholungen nur einmal zählen.
    data = data.loc[valid].drop_duplicates()

    # Widersprüchliche S/R-Werte derselben ID nicht stillschweigend auswählen.
    conflicts = data.loc[data["Sample_ID_IfH"].duplicated(keep=False), "Sample_ID_IfH"]
    if not conflicts.empty:
        raise ValueError(
            f"{device}: widersprüchliche Tobramycin-Ergebnisse für "
            f"{', '.join(conflicts.unique())}"
        )

    labels[device] = data.set_index("Sample_ID_IfH")["tobramycin"]

matched = pd.DataFrame(labels).dropna()
if matched.empty:
    raise ValueError(
        "Keine gemeinsamen Sample-IDs mit S/R-Ergebnissen auf allen drei Geräten."
    )

pairs = [("Phoenix", "VITEK"), ("Phoenix", "MicroScan"), ("VITEK", "MicroScan")]
matrices = {}

for row_device, col_device in pairs:
    matrix = pd.crosstab(matched[row_device], matched[col_device]).reindex(
        index=["S", "R"], columns=["S", "R"], fill_value=0
    )
    matrices[(row_device, col_device)] = matrix

    concordance = (matrix.loc["S", "S"] + matrix.loc["R", "R"]) / len(matched)
    print(
        f"\n{row_device} × {col_device} | n = {len(matched)} | Agreement = {concordance:.1%}"
    )
    print(matrix)


Phoenix × VITEK | n = 217 | Agreement = 62.7%
VITEK     S   R
Phoenix        
S        60   1
R        80  76

Phoenix × MicroScan | n = 217 | Agreement = 93.1%
MicroScan   S    R
Phoenix           
S          49   12
R           3  153

VITEK × MicroScan | n = 217 | Agreement = 59.4%
MicroScan   S   R
VITEK            
S          52  88
R           0  77


csv Export

In [4]:
rows = []

for (row_device, column_device), matrix in matrices.items():
    counts = {
        f"{row_label}_{column_label}": int(matrix.loc[row_label, column_label])
        for row_label in ("S", "R")
        for column_label in ("S", "R")
    }
    n_isolates = sum(counts.values())
    n_agree = counts["S_S"] + counts["R_R"]

    rows.append(
        {
            "row_device": row_device,
            "column_device": column_device,
            **counts,
            "n_isolates": n_isolates,
            "agreement_percent": 100 * n_agree / n_isolates,
        }
    )

pd.DataFrame(rows).to_csv(
    "tobramycin_agreement.csv",
    index=False,
    float_format="%.1f",
)